 # Evaluate training results and choose best models

 How to read results:

| Scenario                              | Precision           | Recall             | F1                | Accuracy        |
|---------------------------------------|---------------------|--------------------|-------------------|-----------------|
| All labels were assign correct        | 1.0                 | 1.0                | 1.0               | 1.0             |
| Some labels correct, some missing     | 1.0                 | <1.0               | <1.0              | <1.0            |
| Some labels correct, some extra       | <1.0                | 1.0                | <1.0              | <1.0            |
| None correct labels                   | 0                   | 0                  | 0                 | 0               |

 ## Libraries

In [ ]:
# Uncomment to install libraries used in this notebook
#!pip install pandas numpy matplotlib seaborn

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

 ## Inputs/outputs directories

In [ ]:
input_dir = "./inputs"
output_dir = "./outputs"
summary_plots_dir = os.path.join(output_dir, "plots", "summary")


os.makedirs(input_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)
os.makedirs(summary_plots_dir, exist_ok=True)


 ## Load training summary file

In [ ]:
summary_path = os.path.join(input_dir, "classification_summary_all.tsv")
df = pd.read_csv(summary_path, sep="\t")
df

In [ ]:
# Display names for models
model_display = {"mlp": "MLP", "rf": "Random Forest", "logreg": "Logistic Regression"}
df["model_disp"] = df["model"].map(model_display)

In [ ]:
# Display names for stratifications
strat_display = {"no_strat": "No Stratification", "minority_strat": "Minority Stratification", "cardinality_strat": "Cardinality Stratification"}
df["strat_mode_disp"] = df["strat_mode"].map(strat_display)

In [ ]:
# Accuracy per dataset and model
plt.figure(figsize=(15,7))
sns.boxplot(data=df, x="dataset", y="Accuracy", hue="model")
plt.title("Mean Accuracy distribution per dataset and model")
plt.ylabel("Mean Accuracy")
plt.xlabel("Dataset")
plt.legend(title="Model")
plt.tight_layout()
plot_path_acc = os.path.join(summary_plots_dir, "mean_accuracy_distribution_per_dataset_and_model.png")
plt.savefig(plot_path_acc)
plt.show()

# F1 per dataset and model
plt.figure(figsize=(15,7))
sns.boxplot(data=df, x="dataset", y="F1", hue="model")
plt.title("Mean F1 distribution per dataset and model")
plt.ylabel("Mean F1")
plt.xlabel("Dataset")
plt.legend(title="Model")
plt.tight_layout()
plot_path_f1 = os.path.join(summary_plots_dir, "mean_f1_distribution_per_dataset_and_model.png")
plt.savefig(plot_path_f1)
plt.show()

In [ ]:
# Mean metrics per group
group_cols = ["dataset", "strat_mode_disp", "model_disp"]
metrics = ["F1", "Accuracy", "Recall", "Precision"]
df_mean = df.groupby(group_cols)[metrics].mean().reset_index()
df_std = df.groupby(group_cols)[metrics].std().reset_index()

In [ ]:
df_mean

In [ ]:
# Heatmap: Mean Accuracy
pivot_acc = df_mean.pivot_table(index=["dataset","model_disp"], columns="strat_mode_disp", values="Accuracy")
pivot_acc.index = [f"{d}: {m}" for d, m in pivot_acc.index]
plt.figure(figsize=(10,8))
sns.heatmap(pivot_acc, annot=True, fmt=".3f", cmap="OrRd")
plt.title("Mean Accuracy per Model by Dataset / Stratification")
plt.ylabel("Dataset / Stratification")
plt.xlabel("Stratification")
plt.tight_layout()
plot_path_heatmap = os.path.join(summary_plots_dir, "mean_accuracy_per_model_by_dataset_per_strat.png")
plt.savefig(plot_path_heatmap)
plt.show()

In [ ]:
best_acc = df.loc[df.groupby(['dataset'])['Accuracy'].idxmax()].reset_index(drop=True)
best_acc["model_strat"] = best_acc["model_disp"] + " - " + best_acc["strat_mode_disp"]
plt.figure(figsize=(14,6))
sns.barplot(data=best_acc, x='dataset', y='Accuracy', hue='model_strat')
for i, row in best_acc.iterrows():
    plt.text(i, row["Accuracy"]+0.01, f"{row['Accuracy']:.3f}", ha="center", fontsize=10)
plt.title("Best Model by Accuracy per Dataset")
plt.ylabel("Accuracy")
plt.xlabel("Dataset")
plt.xticks(rotation=30)
plt.legend(title="Stratisfaction")
plt.tight_layout()
plt.show()

In [ ]:
# Calculate new metric (accuracy * f1):
df["AccF1"] = df["Accuracy"] * df["F1"]

In [ ]:
best_accf1 = df.loc[df.groupby(['dataset'])['AccF1'].idxmax()].reset_index(drop=True)
plt.figure(figsize=(14, 6))
width = 0.3

# Show accuracy and f1 for each best model (chosen by AccF1)
x = np.arange(len(best_accf1))
plt.bar(x - width/2, best_accf1["Accuracy"], width, label="Accuracy")
plt.bar(x + width/2, best_accf1["F1"], width, label="F1 Score")

# Annotate values and model name
for i, row in best_accf1.iterrows():
    plt.text(x[i]-width/2, row["Accuracy"]+0.01, f"{row['Accuracy']:.3f}", ha="center", color="blue", fontsize=10)
    plt.text(x[i]+width/2, row["F1"]+0.01, f"{row['F1']:.3f}", ha="center", color="green", fontsize=10)
    # Add model name, stratification, and seed below the bar with background box
    plt.text(
        x[i], 0.2, f"{row['model_disp']}",
        ha="center", color="black", fontsize=9,
        bbox=dict(facecolor="white", edgecolor="grey", boxstyle="round,pad=0.2", alpha=0.6)
    )
    plt.text(
        x[i], 0.15, f"{row['strat_mode_disp']}",
        ha="center", color="black", fontsize=9,
        bbox=dict(facecolor="white", edgecolor="grey", boxstyle="round,pad=0.2", alpha=0.6)
    )
    plt.text(
        x[i], 0.10, f"seed {row['seed']}",
        ha="center", color="black", fontsize=9,
        bbox=dict(facecolor="white", edgecolor="grey", boxstyle="round,pad=0.2", alpha=0.6)
    )

plt.title("Best Model per Dataset (by Accuracy × F1)")
plt.ylabel("Score")
plt.xlabel("Dataset")
plt.xticks(x, best_accf1["dataset"], rotation=30)
plt.legend()
plt.ylim(0, 1.1)
plt.tight_layout()
plot_path_best = os.path.join(summary_plots_dir, "best_model_per_dataset_acc_f1.png")
plt.savefig(plot_path_best)
plt.show()

In [ ]:
# Save best models (by Accuracy × F1) info as TSV
best_models_path = os.path.join(output_dir, "best_models_accf1.tsv")
best_accf1.to_csv(best_models_path, sep="\t", index=False)

print(f"Best models table saved to: {best_models_path}")

In [ ]:
#Tabular Summary: Best Model Per Dataset/Stratification
print(best_accf1[["dataset","strat_mode_disp","model_disp", "Precision", "Recall", "Accuracy","F1"]].to_string(index=False))

In [ ]:
# For this grouped barplot, we aggregate by dataset + model only for mean Accuracy and F1

mean_scores = df_mean.groupby(["dataset", "model_disp"], as_index=False)[["Accuracy", "F1", "Precision", "Recall"]].mean()

datasets = mean_scores["dataset"].unique()
models = mean_scores["model_disp"].unique()
n_datasets = len(datasets)
n_models = len(models)

plt.figure(figsize=(2.5 * n_datasets, 6))
width = 0.14
x = np.arange(n_datasets)

for i, dataset in enumerate(datasets):
    for j, model in enumerate(models):
        model_data = mean_scores[(mean_scores["dataset"] == dataset) & (mean_scores["model_disp"] == model)]
        acc = model_data["Accuracy"].values[0] if not model_data.empty else np.nan
        f1 = model_data["F1"].values[0] if not model_data.empty else np.nan
        xpos_acc = x[i] + (j * 2 - n_models) * width + width/2
        xpos_f1 = x[i] + (j * 2 - n_models) * width + width*1.5
        # Accuracy bar
        plt.bar(xpos_acc, acc, width, color=f"C{j}", label=f"{model} Accuracy" if i == 0 else None)
        plt.text(xpos_acc, acc+0.01, f"{acc:.3f}", ha="center", color=f"C{j}", fontsize=8)
        # F1 bar
        plt.bar(xpos_f1, f1, width, color=f"C{j}", hatch="///", label=f"{model} F1" if i == 0 else None)
        plt.text(xpos_f1, f1+0.01, f"{f1:.3f}", ha="center", color=f"C{j}", fontsize=8)

plt.title("Mean Accuracy & F1 per Model for Each Dataset (Grouped by Model)")
plt.ylabel("Score")
plt.xlabel("Dataset")
plt.xticks(x, datasets, rotation=30)
plt.ylim(0, 1.1)
handles, labels = plt.gca().get_legend_handles_labels()
unique = dict(zip(labels, handles))
plt.legend(unique.values(), unique.keys(), fontsize=10, ncol=3)
plt.tight_layout()
plot_path = os.path.join(summary_plots_dir, "mean_acc_f1_per_model_per_dataset.png")
plt.savefig(plot_path)
plt.show()

In [ ]:
# Save mean scores data to outputs dir
mean_scores_path = os.path.join(output_dir, "mean_scores_per_model_per_dataset.tsv")
mean_scores.to_csv(mean_scores_path, sep="\t", index=False)

print(f"Mean scores table saved to: {mean_scores_path}")